<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_CMVG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1: Setup and Functions
# ---------------------------------
# This cell imports libraries and defines the necessary calculation functions that will be used in subsequent steps.

import pandas as pd
import numpy as np

def calculate_k_b(sensitivities, correlation):
    """
    Calculates the intra-bucket capital (K_b) based on Article 325f, paragraph 7.
    This function uses the direct summation method for accuracy.
    """
    if len(sensitivities) < 2:
        return np.abs(sensitivities[0]) if len(sensitivities) > 0 else 0

    sum_of_squares = np.sum(np.square(sensitivities))

    cross_terms_sum = 0
    for i in range(len(sensitivities)):
        for j in range(i + 1, len(sensitivities)):
            cross_terms_sum += 2 * correlation * sensitivities[i] * sensitivities[j]

    return np.sqrt(sum_of_squares + cross_terms_sum)

def calculate_total_capital(params, sensitivities_df):
    """
    Calculates the final, across-bucket capital for a given correlation scenario.
    This function incorporates the conservative S_b capping logic.
    """
    # Calculate K_b for each bucket
    ws_b2 = sensitivities_df[sensitivities_df['Bucket'] == 2]['Weighted Sensitivity'].values
    ws_b6 = sensitivities_df[sensitivities_df['Bucket'] == 6]['Weighted Sensitivity'].values
    K_2 = calculate_k_b(ws_b2, params['rho_2'])
    K_6 = calculate_k_b(ws_b6, params['rho_6'])

    # Get S_b for each bucket
    S_2 = sensitivities_df[sensitivities_df['Bucket'] == 2]['Weighted Sensitivity'].sum()
    S_6 = sensitivities_df[sensitivities_df['Bucket'] == 6]['Weighted Sensitivity'].sum()

    # Apply the conservative S_b cap as per Article 325f, paragraph 8
    S_2_capped = max(min(S_2, K_2), -K_2)
    S_6_capped = max(min(S_6, K_6), -K_6)

    # Across-bucket aggregation
    sum_of_k_squares = K_2**2 + K_6**2
    cross_bucket_term = 2 * params['gamma_2_6'] * S_2_capped * S_6_capped

    return np.sqrt(sum_of_k_squares + cross_bucket_term)


print("Libraries and helper functions defined successfully.")


Libraries and helper functions defined successfully.


In [3]:
# Cell 2: Step 1 - Initial Portfolio Data
# ---------------------------------------
data = [
    {'Position ID': 1, 'Bucket': 6, 'Commodity Name': 'UKNBP', 'Option Maturity': '1Y', 'Gross Sensitivity': 74755},
    {'Position ID': 2, 'Bucket': 6, 'Commodity Name': 'NLTTF', 'Option Maturity': '6M', 'Gross Sensitivity': -82734},
    {'Position ID': 3, 'Bucket': 2, 'Commodity Name': 'Brent', 'Option Maturity': '1Y', 'Gross Sensitivity': 311994},
    {'Position ID': 4, 'Bucket': 2, 'Commodity Name': 'Brent', 'Option Maturity': '6M', 'Gross Sensitivity': 1636643},
    {'Position ID': 5, 'Bucket': 2, 'Commodity Name': 'Brent', 'Option Maturity': '6M', 'Gross Sensitivity': -1271914}
]
portfolio_df = pd.DataFrame(data)
print("--- Step 1: Initial Portfolio ---")
portfolio_df


--- Step 1: Initial Portfolio ---


,Position ID,Bucket,Commodity Name,Option Maturity,Gross Sensitivity
0,1,6,UKNBP,1Y,74755
1,2,6,NLTTF,6M,-82734
2,3,2,Brent,1Y,311994
3,4,2,Brent,6M,1636643
4,5,2,Brent,6M,-1271914


In [4]:
# Cell 3: Step 3 - Net Sensitivities
# ---------------------------------------------
print("--- Step 3: Net Sensitivities ---")
net_sensitivities_df = portfolio_df.groupby(['Bucket', 'Commodity Name', 'Option Maturity'])['Gross Sensitivity'].sum().reset_index()
net_sensitivities_df.rename(columns={'Gross Sensitivity': 'Net Sensitivity'}, inplace=True)
net_sensitivities_df

--- Step 3: Net Sensitivities ---


,Bucket,Commodity Name,Option Maturity,Net Sensitivity
0,2,Brent,1Y,311994
1,2,Brent,6M,364729
2,6,NLTTF,6M,-82734
3,6,UKNBP,1Y,74755


In [5]:
# Cell 4: Step 4 - Weighted Sensitivities
# -------------------------------------------------
print("--- Step 4: Weighted Sensitivities ---")
VEGA_RISK_WEIGHT = 1.0
net_sensitivities_df['Weighted Sensitivity'] = net_sensitivities_df['Net Sensitivity'] * VEGA_RISK_WEIGHT
net_sensitivities_df

--- Step 4: Weighted Sensitivities ---


,Bucket,Commodity Name,Option Maturity,Net Sensitivity,Weighted Sensitivity
0,2,Brent,1Y,311994,311994.0
1,2,Brent,6M,364729,364729.0
2,6,NLTTF,6M,-82734,-82734.0
3,6,UKNBP,1Y,74755,74755.0


In [6]:
# Cell 5: Step 5 - Intra-Bucket Correlation Coefficients (Medium Scenario)
# ---------------------------------------------------------------------
print("--- Step 5: Intra-Bucket Correlation Coefficients (Medium Scenario) ---")
print("Note: Delta component based on BCBS MAR21.94 FAQ (commodity type only).")

# Bucket 2
rho_delta_cty_2 = 1.00 # Identical commodities (Brent vs Brent)
t_k, t_l = 1.0, 0.5
rho_opt_maturity = np.exp(-0.01 * (abs(t_k - t_l) / min(t_k, t_l)))
rho_2 = rho_delta_cty_2 * rho_opt_maturity

# Bucket 6
rho_delta_cty_6 = 0.65 # Different commodities (from Table 10)
rho_6 = rho_delta_cty_6 * rho_opt_maturity

medium_params = {'rho_2': rho_2, 'rho_6': rho_6}
medium_corr_df = pd.DataFrame([
    {'Parameter': 'Final Intra-Bucket Correlation (rho_2)', 'Value': rho_2},
    {'Parameter': 'Final Intra-Bucket Correlation (rho_6)', 'Value': rho_6}
]).set_index('Parameter')
medium_corr_df

--- Step 5: Intra-Bucket Correlation Coefficients (Medium Scenario) ---
Note: Delta component based on BCBS MAR21.94 FAQ (commodity type only).


,Value
Parameter,
Final Intra-Bucket Correlation (rho_2),0.990050
Final Intra-Bucket Correlation (rho_6),0.643532


In [7]:
# Cell 6: Step 7 - Intra-Bucket Aggregation (Medium Scenario)
# -----------------------------------------------------------
print("--- Step 7: Intra-Bucket Capital (Medium Scenario) ---")
ws_b2 = net_sensitivities_df[net_sensitivities_df['Bucket'] == 2]['Weighted Sensitivity'].values
ws_b6 = net_sensitivities_df[net_sensitivities_df['Bucket'] == 6]['Weighted Sensitivity'].values

K2_med = calculate_k_b(ws_b2, medium_params['rho_2'])
K6_med = calculate_k_b(ws_b6, medium_params['rho_6'])

k_b_medium_df = pd.DataFrame([
    {'Bucket': 2, 'K_b (Medium)': K2_med},
    {'Bucket': 6, 'K_b (Medium)': K6_med}
]).set_index('Bucket')
k_b_medium_df.style.format('{:,.0f}')

--- Step 7: Intra-Bucket Capital (Medium Scenario) ---


,K_b (Medium)
Bucket,
2,"675,048"
6,"66,881"


In [8]:
# Cell 7: Step 8 - Across-Bucket Aggregation (Medium Scenario)
# ------------------------------------------------------------
print("--- Step 8: Across-Bucket Capital (Medium Scenario) ---")
gamma_2_6 = 0.20
medium_params['gamma_2_6'] = gamma_2_6

capital_medium = calculate_total_capital(medium_params, net_sensitivities_df)
print(f"Medium Scenario Total Capital: {capital_medium:,.0f}")


--- Step 8: Across-Bucket Capital (Medium Scenario) ---
Medium Scenario Total Capital: 676,763


In [9]:
# Cell 8: Step 9 (Part 1) - Derive High and Low Correlation Scenarios
# -------------------------------------------------------------------
print("--- Step 9: Derived Scenario Coefficients ---")
high_params = {
    'rho_2': min(rho_2 * 1.25, 1.0),
    'rho_6': min(rho_6 * 1.25, 1.0),
    'gamma_2_6': min(gamma_2_6 * 1.25, 1.0)
}
low_params = {
    'rho_2': max(2 * rho_2 - 1, 0.75 * rho_2),
    'rho_6': max(2 * rho_6 - 1, 0.75 * rho_6),
    'gamma_2_6': max(2 * gamma_2_6 - 1, 0.75 * gamma_2_6)
}
derived_params_df = pd.DataFrame([
    {'Scenario': 'Medium', 'rho_2': medium_params['rho_2'], 'rho_6': medium_params['rho_6'], 'gamma_2_6': medium_params['gamma_2_6']},
    {'Scenario': 'High', 'rho_2': high_params['rho_2'], 'rho_6': high_params['rho_6'], 'gamma_2_6': high_params['gamma_2_6']},
    {'Scenario': 'Low', 'rho_2': low_params['rho_2'], 'rho_6': low_params['rho_6'], 'gamma_2_6': low_params['gamma_2_6']}
]).set_index('Scenario')
derived_params_df

--- Step 9: Derived Scenario Coefficients ---


,rho_2,rho_6,gamma_2_6
Scenario,,,
Medium,0.99005,0.643532,0.20
High,1.00000,0.804415,0.25
Low,0.98010,0.482649,0.15


In [10]:
# Cell 9: Step 9 (Part 2) - Calculate All Scenario Capitals
# ---------------------------------------------------------
print("--- Scenario Capital Results ---")
capital_high = calculate_total_capital(high_params, net_sensitivities_df)
capital_low = calculate_total_capital(low_params, net_sensitivities_df)

results_data = [
    {'Scenario': 'Medium', 'Total Capital': capital_medium},
    {'Scenario': 'High', 'Total Capital': capital_high},
    {'Scenario': 'Low', 'Total Capital': capital_low}
]
results_df = pd.DataFrame(results_data).set_index('Scenario')
results_df.style.format('{:,.0f}')

--- Scenario Capital Results ---


,Total Capital
Scenario,
Medium,"676,763"
High,"676,563"
Low,"676,961"


In [11]:
# Cell 10: Step 10 - Final Charge Calculation
# -------------------------------------------
print("--- Step 10: Final Charge Calculation ---")
final_capital_charge = max(capital_medium, capital_high, capital_low)
final_result_df = pd.DataFrame([{'Final Commodity Vega Capital Requirement': final_capital_charge}])
final_result_df.style.format('{:,.0f}')

--- Step 10: Final Charge Calculation ---


,Final Commodity Vega Capital Requirement
0,"676,961"
